# Final Project: Build a RAG-Powered Knowledge Assistant


## Project Overview

In this project, you will build a complete **Retrieval-Augmented Generation (RAG) system** from scratch. Your system will serve as an intelligent knowledge assistant that can answer questions based on a collection of documents you provide.

This project brings together everything you have learned in this chapter:
- Document chunking strategies
- Text embeddings with Sentence Transformers
- Vector storage and search with ChromaDB
- Question answering with Hugging Face models
- Building end-to-end RAG pipelines


## Choose Your Scenario

Select **ONE** of the following scenarios for your project:

### Option A: Customer Support Assistant
Build a customer support chatbot for a fictional company. Your knowledge base should include:
- Product/service descriptions
- Pricing and plans
- FAQs and troubleshooting
- Contact and support information
- Policies (returns, refunds, etc.)

### Option B: Study Guide Assistant
Build a study assistant for a subject you are learning. Your knowledge base should include:
- Key concepts and definitions
- Explanations of important topics
- Examples and use cases
- Common questions and answers
- Summary notes

### Option C: Personal Interest Assistant
Build an assistant for a hobby, interest, or domain you know well. Examples:
- A cooking assistant with recipes and techniques
- A gaming wiki assistant
- A travel guide for a city or region
- A fitness/workout assistant
- Any other domain you are passionate about


## Project Requirements

Your RAG system must meet the following requirements:

### Knowledge Base Requirements
- [ ] Minimum of **8 documents** covering different aspects of your chosen topic
- [ ] Each document should be at least **150 words**
- [ ] Documents should cover diverse subtopics within your domain
- [ ] Content should be factual and specific (names, numbers, details)

### Technical Requirements
- [ ] Implement document chunking with configurable chunk size and overlap
- [ ] Store chunks in ChromaDB with appropriate metadata
- [ ] Create a RAG pipeline class that handles retrieval and generation
- [ ] Implement confidence-based response handling
- [ ] Include source attribution in responses

### Testing Requirements
- [ ] Test your system with at least **15 different questions**
- [ ] Include questions of varying difficulty (simple facts, comparisons, multi-part)
- [ ] Include at least **3 edge case questions** (questions not answerable from your knowledge base)
- [ ] Document the results of all tests

### Documentation Requirements
- [ ] Explain your scenario choice and knowledge base design
- [ ] Justify your chunking strategy
- [ ] Analyze your system's strengths and weaknesses
- [ ] Suggest improvements for a production version

---

## Part 1: Setup and Configuration

Let's start by installing and importing the required libraries.

In [ ]:
# Install required libraries
!pip install transformers torch sentence-transformers chromadb --quiet

print("✅ Installation complete!")

In [1]:
# Import all required libraries
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from typing import List, Dict, Optional
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n All imports successful!")

PyTorch version: 2.12.1+cpu
CUDA available: False

 All imports successful!


In [2]:
from pathlib import Path

# Directory containing the knowledge base
DOCS_DIR = Path("docs")

def load_markdown_documents(docs_dir=DOCS_DIR):
    """
    Load all Markdown documents from the knowledge base.

    Returns:
        List[dict]: Each document contains:
            - source: filename
            - text: document contents
    """
    documents = []

    for file_path in sorted(docs_dir.glob("*.md")):
        with open(file_path, "r", encoding="utf-8") as f:
            documents.append({
                "source": file_path.name,
                "text": f.read()
            })

    return documents


# Load the knowledge base
documents = load_markdown_documents()

print("=" * 50)
print(f"Loaded {len(documents)} knowledge base documents\n")

for doc in documents:
    print(f"📄 {doc['source']} ({len(doc['text'])} characters)")

Loaded 8 knowledge base documents

📄 01_ml_fundamentals.md (3137 characters)
📄 02_data_preprocessing.md (9213 characters)
📄 03_feature_enginering_encoding.md (0 characters)
📄 04_supervised_learning.md (7497 characters)
📄 05_model_evaluation.md (10027 characters)
📄 06_nueral_networks_deep_learning.md (10765 characters)
📄 07_computer_vision.md (8845 characters)
📄 08_nlp_rag.md (9554 characters)


In [3]:
# Validate the external Markdown knowledge base in docs/
print("KNOWLEDGE BASE VALIDATION")
print("=" * 50)
print(f"Total documents: {len(documents)}")

total_words = 0
for doc in documents:
    word_count = len(doc["text"].split())
    total_words += word_count
    print(f"- {doc['source']}: {word_count} words")

avg_words = total_words // len(documents) if documents else 0
print(f"\nAverage words per document: {avg_words}")

if len(documents) >= 8:
    print("\nRequirement met: At least 8 documents available in docs/.")
else:
    print(f"\nYou need at least 8 documents. Currently have: {len(documents)}")

KNOWLEDGE BASE VALIDATION
Total documents: 8
- 01_ml_fundamentals.md: 458 words
- 02_data_preprocessing.md: 1218 words
- 03_feature_enginering_encoding.md: 0 words
- 04_supervised_learning.md: 1119 words
- 05_model_evaluation.md: 1463 words
- 06_nueral_networks_deep_learning.md: 1565 words
- 07_computer_vision.md: 1284 words
- 08_nlp_rag.md: 1387 words

Average words per document: 1061

Requirement met: At least 8 documents available in docs/.


---

## Part 2: Project Declaration

Before building, declare your project details. This helps you plan and will be part of your submission.

In [ ]:
# ============================================================
# TODO: Fill in your project details
# ============================================================

PROJECT_INFO = {
    "scenario": "Study Guide",
    "topic": "Machine Learning Interview Preparation Assistant",
    "description": (
        "A Retrieval-Augmented Generation (RAG) knowledge assistant that answers "
        "machine learning and data science interview questions using a curated "
        "knowledge base covering machine learning fundamentals, data preprocessing, "
        "feature engineering, supervised learning, model evaluation, neural networks, "
        "computer vision, natural language processing, embeddings, and RAG systems."
    ),
    "target_users": (
        "Junior Data Scientists, TripleTen students, bootcamp graduates, "
        "career changers, and professionals preparing for machine learning "
        "technical interviews."
    )
}

# Print your declaration
print("PROJECT DECLARATION")
print("=" * 50)
for key, value in PROJECT_INFO.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

---

## Part 3: Create Your Knowledge Base

This is the foundation of your RAG system. Create comprehensive, well-organized documents for your chosen scenario.

### Guidelines for Good Knowledge Base Documents:
- **Be specific**: Include names, numbers, dates, and concrete details
- **Be comprehensive**: Cover the topic thoroughly
- **Be organized**: Structure information logically
- **Be accurate**: Ensure facts are correct and consistent across documents

# Part 3: Knowledge Base

Instead of embedding documents directly inside this notebook, this project stores the knowledge base as individual Markdown files located in the `docs/` directory.

This approach provides several advantages:

- Better project organization
- Easier maintenance
- Cleaner notebook
- Scalable knowledge base
- Professional repository structure

The Markdown documents are automatically loaded in Part 1 using a custom document loader.

# ============================================================
# TODO: Create your knowledge base with at least 8 documents
# ============================================================



---

## Part 4: Document Chunking

Implement a chunking function and process your knowledge base. You should experiment with chunk size to find what works best for your content.

In [7]:
# ============================================================
# TODO: Implement your chunking function
# ============================================================

def chunk_document(
    text: str,
    chunk_size: int = 900,
    chunk_overlap: int = 150
) -> List[str]:
    """
    Split one document into overlapping chunks while preserving sentence boundaries.
    """
    if not text or not text.strip():
        return []

    clean_text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", clean_text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if len(current_chunk) + len(sentence) + 1 <= chunk_size:
            current_chunk = f"{current_chunk} {sentence}".strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)

            if chunks and chunk_overlap > 0:
                overlap_text = chunks[-1][-chunk_overlap:]
                current_chunk = f"{overlap_text} {sentence}".strip()
            else:
                current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


def chunk_documents(
    documents: List[Dict],
    chunk_size: int = 900,
    chunk_overlap: int = 150
) -> List[Dict]:
    """
    Chunk all loaded Markdown documents from docs/.
    """
    chunk_records = []
    for doc in documents:
        doc_chunks = chunk_document(
            text=doc["text"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for i, chunk_text in enumerate(doc_chunks):
            chunk_records.append({
                "id": f"{doc['source'].replace('.md', '')}_chunk_{i}",
                "text": chunk_text,
                "source": doc["source"],
                "chunk_index": i,
                "title": doc["source"].replace("_", " ").replace(".md", "")
            })

    return chunk_records


print("Chunking functions ready. Run the next cell to chunk all documents.")

Chunking functions ready. Run the next cell to chunk all documents.


In [8]:
# ============================================================
# Process all documents into chunks with metadata
# ============================================================

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# Stage 2: Chunk documents from the docs/ source of truth
chunks = chunk_documents(
    documents=documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# Prepare vectors payloads from the chunk list for ChromaDB
all_chunks = [chunk["text"] for chunk in chunks]
all_metadatas = [
    {
        "source": chunk["source"],
        "title": chunk["title"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in chunks
]
all_ids = [f"{chunk['source']}_chunk_{chunk['chunk_index']}" for chunk in chunks]

avg_chunk_len = int(sum(len(c["text"]) for c in chunks) / len(chunks)) if chunks else 0

print("CHUNKING RESULTS")
print("=" * 60)
print(f"Number of source documents: {len(documents)}")
print(f"Number of chunks: {len(chunks)}")
print(f"Average chunk length: {avg_chunk_len} characters")

print("\nPreview of first 3 chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    preview = chunk["text"][:200].replace("\n", " ")
    print(f"\n{i}. Source: {chunk['source']} | Chunk Index: {chunk['chunk_index']}")
    print(f"   {preview}...")

CHUNKING RESULTS
Number of source documents: 8
Number of chunks: 86
Average chunk length: 798 characters

Preview of first 3 chunks:

1. Source: 01_ml_fundamentals.md | Chunk Index: 0
   # Machine Learning Fundamentals & Data Science Workflow Machine learning is a branch of artificial intelligence where computers learn patterns from data instead of being explicitly programmed with fix...

2. Source: 01_ml_fundamentals.md | Chunk Index: 1
   ke? What data is available? What are the risks of being wrong? This matters because the best technical model is not always the best business solution. After defining the problem, the next step is data...

3. Source: 01_ml_fundamentals.md | Chunk Index: 2
   gression predicts continuous numeric values, such as house price or delivery time. In unsupervised learning, the data does not include a target label. Common unsupervised tasks include clustering cust...


In [9]:
# Part 4 schema validation (chunks only)
required_keys = {"id", "text", "source", "chunk_index", "title"}

print("PART 4 CHUNK SCHEMA VALIDATION")
print("=" * 50)

chunks_exists = "chunks" in globals()
print(f"1) chunks exists: {chunks_exists}")

if chunks_exists:
    print(f"2) len(chunks) == 86: {len(chunks) == 86} (actual: {len(chunks)})")

    missing_key_count = 0
    for idx, chunk in enumerate(chunks):
        missing = required_keys - set(chunk.keys())
        if missing:
            missing_key_count += 1
            print(f"Missing keys in chunk {idx}: {sorted(missing)}")

    print(f"3) every chunk has required keys: {missing_key_count == 0}")
    print(f"4) first chunk dictionary keys: {sorted(chunks[0].keys()) if chunks else []}")
else:
    print("2) len(chunks) == 86: False (chunks not available)")
    print("3) every chunk has required keys: False (chunks not available)")
    print("4) first chunk dictionary keys: []")

PART 4 CHUNK SCHEMA VALIDATION
1) chunks exists: True
2) len(chunks) == 86: True (actual: 86)
3) every chunk has required keys: True
4) first chunk dictionary keys: ['chunk_index', 'id', 'source', 'text', 'title']


### Chunking Strategy Justification

**TODO:** In the cell below, explain your chunking decisions:

## Chunking Strategy Justification

### Why did you choose this chunk size?

I selected a chunk size of 900 characters because the knowledge base documents contain detailed interview explanations with definitions, examples, and comparisons. This size is large enough to preserve meaningful context while still keeping each chunk focused for semantic retrieval.

### Why did you choose this overlap amount?

I used a 150-character overlap so related information is not lost when text is split between chunks. The overlap helps preserve continuity between neighboring chunks and improves retrieval quality for questions that depend on surrounding context.

### Did you make any modifications to handle your specific content type?

Yes. Instead of storing documents directly inside the notebook, this project uses Markdown files in the `docs/` folder as the source of truth. The chunking pipeline preserves metadata for each chunk, including the source document, chunk index, title, and unique chunk ID. This makes the RAG pipeline easier to debug, maintain, and explain.

---

## Part 5: Vector Database Setup

Create your ChromaDB collection and add all chunks with embeddings.

In [ ]:
# ============================================================
# Set up ChromaDB and create your collection
# ============================================================

## Stage 3: Generate embeddings for chunk text using Sentence Transformers
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

## Stage 4: Store chunk vectors in ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="ml_interview_rag_chunks",
    embedding_function=embedding_function,
    metadata={
        "description": "Chunked Markdown docs for ML interview RAG assistant",
        "source_of_truth": "docs/"
    }
)

print("Collection created!")
print(f"Ready to ingest {len(chunks)} chunks from {len(documents)} source documents.")

In [ ]:
# ============================================================
# Add all chunks to the collection
# ============================================================

## Stage 4 (continued): Persist chunk vectors and metadata in ChromaDB
collection.add(
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"✅ Added {collection.count()} chunks to the database")

In [ ]:
## Stage 5: Retrieve relevant chunks for a sample query
test_query = "What is the difference between precision and recall?"

results = collection.query(
    query_texts=[test_query],
    n_results=3
)

print(f"Test Query: \"{test_query}\"")
print("\nTop 3 Retrieved Chunks:")
for i in range(len(results['documents'][0])):
    print(f"\n{i+1}. Source: {results['metadatas'][0][i].get('source', 'Unknown')}")
    print(f"   Preview: {results['documents'][0][i][:150]}...")

---

## Part 6: Build the RAG Pipeline

Create a complete RAG pipeline class with all required functionality.

In [ ]:
## Stage 6: Generate the final answer with a QA model over retrieved context
qa_model = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("QA model loaded!")

In [ ]:
# ============================================================
# TODO: Implement your RAG Pipeline class
# ============================================================

class RAGAssistant:
    """
    A complete RAG-powered knowledge assistant.

    Your implementation must include:
    1. Document retrieval from ChromaDB
    2. Answer generation using the QA model
    3. Confidence-based response handling
    4. Source attribution
    """

    def __init__(self, collection, qa_model, confidence_threshold: float = 0.3):
        """
        Initialize the RAG assistant.

        Args:
            collection: ChromaDB collection with embedded documents
            qa_model: Hugging Face QA pipeline
            confidence_threshold: Minimum confidence to provide a direct answer
        """
        self.collection = collection
        self.qa_model = qa_model
        self.confidence_threshold = confidence_threshold

    def retrieve(self, query: str, n_results: int = 3) -> Dict:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The user's question
            n_results: Number of documents to retrieve

        Returns:
            Dict containing documents, metadatas, and ids
        """
        # TODO: Implement retrieval
        pass

    def generate_answer(self, question: str, context: str) -> Dict:
        """
        Generate an answer from the given context.

        Args:
            question: The user's question
            context: Retrieved context to answer from

        Returns:
            Dict with answer and confidence score
        """
        # TODO: Implement answer generation
        pass

    def ask(self, question: str, n_results: int = 3) -> Dict:
        """
        Complete RAG pipeline: retrieve and generate answer.

        This method should:
        1. Retrieve relevant documents
        2. Combine them into context
        3. Generate an answer
        4. Handle low confidence appropriately
        5. Include source attribution

        Args:
            question: The user's question
            n_results: Number of documents to retrieve

        Returns:
            Dict with:
            - question: Original question
            - answer: Generated answer (or fallback message)
            - confidence: Confidence score
            - sources: List of source documents used
            - is_confident: Boolean indicating if confidence met threshold
        """
        # TODO: Implement the complete pipeline
        pass

    def format_response(self, result: Dict) -> str:
        """
        Format the result into a user-friendly response string.

        Args:
            result: Output from the ask() method

        Returns:
            Formatted string response
        """
        # TODO: Implement response formatting
        pass


# Create your assistant
assistant = RAGAssistant(collection, qa_model, confidence_threshold=0.3)

print("✅ RAG Assistant created!")

In [ ]:
# ============================================================
# Test your assistant with a few questions
# ============================================================

test_questions = [
    "Replace with a question about your knowledge base",
    "Replace with another question",
    "Replace with a third question",
]

print("INITIAL TESTING")
print("=" * 60)
for question in test_questions:
    result = assistant.ask(question)
    print(f"\nQ: {question}")
    print(assistant.format_response(result))

---

## Part 7: Comprehensive Testing

Test your system thoroughly with at least 15 questions covering different types and difficulty levels.

In [ ]:
# ============================================================
# TODO: Create your comprehensive test suite
# ============================================================

# Organize your test questions by category
test_suite = {
    "simple_facts": [
        # Questions with straightforward, single-fact answers
        # Example: "What is the price of X?" "When does Y open?"
        "Question 1",
        "Question 2",
        "Question 3",
        "Question 4",
        "Question 5",
    ],

    "detailed_explanations": [
        # Questions requiring more detailed answers
        # Example: "How does X work?" "What are the features of Y?"
        "Question 6",
        "Question 7",
        "Question 8",
        "Question 9",
        "Question 10",
    ],

    "comparison_or_complex": [
        # More complex questions
        # Example: "What's the difference between X and Y?"
        "Question 11",
        "Question 12",
    ],

    "edge_cases": [
        # Questions NOT answerable from your knowledge base
        # These test your fallback handling
        "Question 13 (not in KB)",
        "Question 14 (not in KB)",
        "Question 15 (not in KB)",
    ]
}

In [ ]:
# ============================================================
# Run comprehensive tests and collect results
# ============================================================

test_results = []

print("COMPREHENSIVE TEST RESULTS")
print("=" * 70)

for category, questions in test_suite.items():
    print(f"\n\n{'='*70}")
    print(f"CATEGORY: {category.upper().replace('_', ' ')}")
    print("=" * 70)

    for question in questions:
        result = assistant.ask(question)

        # Store result for analysis
        test_results.append({
            "category": category,
            "question": question,
            "answer": result["answer"],
            "confidence": result["confidence"],
            "is_confident": result["is_confident"],
            "sources": result["sources"]
        })

        # Display result
        confidence_indicator = "✅" if result["is_confident"] else "⚠️"
        print(f"\n{confidence_indicator} Q: {question}")
        print(f"   A: {result['answer']}")
        print(f"   Confidence: {result['confidence']:.2%} | Sources: {', '.join(result['sources'][:2])}")

In [ ]:
# ============================================================
# Analyze test results
# ============================================================

print("\n" + "=" * 70)
print("TEST RESULTS SUMMARY")
print("=" * 70)

total_questions = len(test_results)
confident_answers = sum(1 for r in test_results if r["is_confident"])
avg_confidence = sum(r["confidence"] for r in test_results) / total_questions

print(f"\nTotal questions tested: {total_questions}")
print(f"Confident answers: {confident_answers} ({confident_answers/total_questions:.1%})")
print(f"Low confidence answers: {total_questions - confident_answers}")
print(f"Average confidence: {avg_confidence:.2%}")

# By category
print("\nResults by category:")
for category in test_suite.keys():
    category_results = [r for r in test_results if r["category"] == category]
    cat_confident = sum(1 for r in category_results if r["is_confident"])
    cat_avg = sum(r["confidence"] for r in category_results) / len(category_results)
    print(f"  {category}: {cat_confident}/{len(category_results)} confident, avg confidence: {cat_avg:.2%}")

---

## Part 8: Analysis and Reflection

Analyze your system's performance and document your findings.

### 8.1 Performance Analysis

*Double-click to edit this cell*

**Which types of questions did your system handle best? Why?**

(Your analysis here)

**Which types of questions did your system struggle with? Why?**

(Your analysis here)

**Did the edge case questions correctly trigger low-confidence responses?**

(Your analysis here)

### 8.2 Strengths and Weaknesses

*Double-click to edit this cell*

**List 3 strengths of your RAG system:**

1.
2.
3.

**List 3 weaknesses or limitations:**

1.
2.
3.

### 8.3 Improvement Recommendations

*Double-click to edit this cell*

**If you were to deploy this as a production system, what improvements would you make?**

Consider:
- Knowledge base improvements
- Chunking strategy changes
- Retrieval enhancements
- Generation improvements
- User experience features


---

## Part 9: Demo Showcase

Create a polished demo of your assistant in action.

In [ ]:
# ============================================================
# Create a demo showcasing your assistant
# ============================================================

def run_demo(assistant, demo_questions: List[str]):
    """
    Run a polished demo of the RAG assistant.
    """
    print("\n" + "="*70)
    print(f"{PROJECT_INFO['topic']} - Knowledge Assistant Demo")
    print(f"   {PROJECT_INFO['description']}")
    print("="*70)

    for i, question in enumerate(demo_questions, 1):
        print(f"\n{'─'*70}")
        print(f"User Question {i}:")
        print(f"   \"{question}\"")
        print()

        result = assistant.ask(question)

        print(f"Assistant Response:")
        print(f"   {result['answer']}")
        print()
        print(f"   Sources: {', '.join(result['sources'][:2])}")
        print(f"   Confidence: {result['confidence']:.1%}")

    print(f"\n{'='*70}")
    print("Demo complete!")
    print("="*70)


# Select 5 of your best questions for the demo
demo_questions = [
    # TODO: Choose 5 questions that showcase your system well
    "Demo question 1",
    "Demo question 2",
    "Demo question 3",
    "Demo question 4",
    "Demo question 5",
]

run_demo(assistant, demo_questions)

---

## Submission Checklist

Before submitting, verify you have completed all requirements:

### Knowledge Base
- [ ] At least 8 documents in your knowledge base
- [ ] Each document is at least 150 words
- [ ] Documents cover diverse subtopics
- [ ] Content includes specific, factual information

### Technical Implementation
- [ ] Chunking function implemented with configurable parameters
- [ ] ChromaDB collection created and populated
- [ ] RAG pipeline class with all required methods
- [ ] Confidence-based response handling implemented
- [ ] Source attribution included in responses

### Testing
- [ ] At least 15 test questions
- [ ] Questions organized by category/difficulty
- [ ] At least 3 edge case questions included
- [ ] All test results documented

### Documentation
- [ ] Project declaration completed (Part 2)
- [ ] Chunking strategy justified (Part 4)
- [ ] Performance analysis completed (Part 8.1)
- [ ] Strengths and weaknesses identified (Part 8.2)
- [ ] Improvement recommendations provided (Part 8.3)
- [ ] Lessons learned documented (Part 8.4)

### Demo
- [ ] Demo showcase with 5 well-chosen questions
- [ ] All code cells executed with visible output



## Congratulations!

You have built a complete RAG-powered knowledge assistant from scratch!

### What You Demonstrated:

1. Creating and organizing a knowledge base
2. Implementing document chunking strategies
3. Using vector databases for semantic search
4. Building end-to-end RAG pipelines
5. Handling edge cases and uncertainty
6. Testing and analyzing system performance
7. Documenting technical decisions and learnings


### Submission Instructions

1. Ensure all code cells have been executed
2. Verify all text cells are filled in
3. Save the notebook (File → Save)
4. Download as .ipynb (File → Download → Download .ipynb)
5. Rename the file to: `RAG_Project_[YourName].ipynb`